# Healthy sampling geometry (and a red-noise tip)

Notebook **01** showed how to sample timing parameters. Before you commit to a
long NUTS run, you should ask: *is the geometry of this target actually
tractable away from the mode?*

When every timing axis is sampled, the likelihood is linearized around an
**expansion point** (usually the par file) and each axis has a **chart** from
physical δ to prior-normal z. A wide uniform chart on a sensitive parameter
(spin frequency is the classic example) can look fine near the mode and explode
in the tails — exactly where HMC explores. This notebook shows how to catch that
*before* sampling.

You will:

1. **Refine the expansion** — optionally move the linearization to the
   conditional timing maximum (marginal likelihood, red noise integrated).
2. **Certify the joint geometry** — probe the exact NumPyro target off-mode;
   inspect residual remainder and Hessian eigenvalues. Nothing here runs inside
   NUTS; `passed=False` is a modeling signal, not a solver crash.
3. **Fix a common modeling mistake** — spin/position axes that are physically
   linear in phase but sit on wide uniform charts by default; declaring them
   `identically_linear` swaps in globally affine charts and collapses the
   off-mode curvature.
4. **Pivot-amplitude red noise** — a small parameterization tip: sample
   amplitude at a sensitivity-weighted frequency instead of 1/yr to decorrelate
   amplitude and slope.

Simulated data. Like notebook **01**, the pulsar is built with MetaPulsar —
currently the only `TimingPulsar` implementation `nltiming` can bind to. Run in
an environment with MetaPulsar + JUG (e.g. the MetaPulsar devcontainer). Read
**01** first if the inference-plan / chart vocabulary is new.


In [1]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import tempfile
from pathlib import Path

import numpy as np

import discovery as ds
from discovery import transport as dst
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import (
    NonLinearTimingModel,
    box_hyper_probe_points, certify_joint_geometry, transport_center_report,
    write_geometry_report, read_geometry_report, refine_timing_expansion,
)
from nltiming.sampling import numpyro as N

ds.config(kernels="metamath")
configure_logging(level="WARNING")
workdir = Path(tempfile.mkdtemp(prefix="nlt_geom_"))


## 0. Setup: pulsar, full-basis context, reference noise

Same kind of simulated pulsar as notebook 01. We use `inference="all"` (sample
every timing axis) and a frozen white-noise reference for the joint transport.
The red-noise hyper point `center` is where we will refine and certify.


In [2]:
import pint.config
from pint.models import get_model
from pint.simulation import make_fake_toas_uniform

np.random.seed(7)
model = get_model(pint.config.examplefile("NGC6440E.par"))
toas = make_fake_toas_uniform(
    startMJD=53400, endMJD=56000, ntoas=150, model=model, obs="gbt",
    error=1.0, add_noise=True,
)
(workdir / "d.par").write_text(model.as_parfile())
toas.write_TOA_file(str(workdir / "d.tim"), format="tempo2")
mp = create_metapulsar(
    {"demo": [{"par": str(workdir / "d.par"), "tim": str(workdir / "d.tim"),
               "timing_package": "pint"}]},
    use_pulse_numbers="no",
)

ctx = NonLinearTimingModel(
    engines="jug", inference="all", name="timing"
).for_pulsar(mp)
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
reference = dst.reference_noise_frozen(
    ds.makenoise_measurement_simple(mp, nd), nd, description="WN MPE")
center = {f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.0}
print("sampled timing axes:", len(ctx.sampled))


[!] Unrecognized par file parameters (ignored by JUG): DMDATA, NE_SW1
[FITTER] noise_config was None, auto-detecting from par file
sampled timing axes: 8


## 1. Refine the linearization expansion

The timing delay is linearized around an expansion point in z-space (by default,
the par-file reference → δ = 0). If that point is a poor approximation of the
posterior mode, the local geometry that NUTS sees is skewed.

`refine_timing_expansion` maximizes a **marginal** timing objective (red noise
analytically integrated; `joint=False` Discovery signals) with SciPy L-BFGS-B and
returns a context re-linearized there. A well-fit par file is often already at
the timing MPE, so the shift can be nearly zero — that is fine.


In [3]:
psl_marginal = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx.discovery_signals(joint=False),
])
objective = N.conditional_timing_potential(psl_marginal, ctx, fixed={**nd, **center})
refined = refine_timing_expansion(ctx, negative_log_target_z=objective)
ctx_cert = refined.context
z0 = np.asarray(ctx.linearization.sampled_z_expansion)
z1 = np.asarray(ctx_cert.linearization.sampled_z_expansion)
print("converged:", refined.converged, " |d z_e|:", float(np.linalg.norm(z1 - z0)))


converged: False  |d z_e|: 0.0


## 2. Certify the joint target geometry (before sampling)

Build the joint NumPyro model and probe it at a small set of red-noise hyper
points around `center`. The certifier reports how well the local linearization
matches the exact residual, and whether the Hessian in the sampler coordinate
looks like the identity (what HMC wants).

`passed` is **advisory** — it never gates `N.nuts`. A failure with huge residual
RMS or a Hessian eigenvalue far from 1 means the geometry is bad for sampling;
fix the model (charts, expansion, reference noise), do not raise the tree depth.


In [4]:
psl_joint = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_cert.discovery_signals(joint=True),
])
jm = N.joint_model(psl_joint, ctx_cert, reference_noise=reference, fixed=nd)

bounds = {f"{mp.name}_rednoise_log10_A": (-18.0, -11.0),
          f"{mp.name}_rednoise_gamma": (0.0, 7.0)}
hyper_points = box_hyper_probe_points(center, bounds)[:3]
report = certify_joint_geometry(jm, ctx_cert, hyper_points=hyper_points)
print("passed :", report.passed)
print("rms    :", round(report.max_residual_remainder_rms, 4),
      " max std/TOA:", round(report.max_residual_remainder_standardized_toa, 3))
print("xi grad:", round(report.max_xi_gradient_inf_norm, 4),
      " H eig  :", round(report.xi_hessian_eigen_min, 3), "..",
      round(report.xi_hessian_eigen_max, 3))
print("id spread:", round(report.max_conditional_identity_spread, 4))
for f in report.failures:
    print("  FAIL:", f)


passed : False
rms    : 625.0731  max std/TOA: 1474.542
xi grad: 543885.4311  H eig  : -4105.997 .. 2910467.91
id spread: 29301842.3738
  FAIL: residual_remainder_rms=625 > 0.1
  FAIL: residual_remainder_max_standardized_toa=1.47e+03 > 1.0
  FAIL: xi_gradient_inf_norm=5.44e+05 > 0.2
  FAIL: xi_hessian_eigen_min=-4.11e+03 < 0.5
  FAIL: xi_hessian_eigen_max=2.91e+06 > 2.0
  FAIL: xi_eta_cross_operator_norm=6.77e+05 > 0.25
  FAIL: conditional_identity_spread=2.93e+07 > 0.1


## 2b. The modeling fix: declare linear axes `identically_linear`

The certification above almost certainly **failed hard** — residual RMS in the
hundreds, a Hessian eigenvalue near 10⁶, maybe even a *negative* eigenvalue.
That is the certifier doing its job: the geometry is broken away from the mode.

**Why?** `F0` and `F1` are physically linear in pulse phase, but by default
`nltiming` only auto-certifies a conservative set (DM family, jumps, offsets,
…). Spin and sky position therefore get wide uniform **PIT** charts
(`prior_pit`). A unit-step probe in the sampler coordinate then reaches deep
into those prior tails, where spin sensitivity is enormous and the exact timing
residual explodes.

The fix is a modeling declaration, not a sampler tweak. Tell `nltiming` that
`F0`/`F1`/`RAJ`/`DECJ` are identically linear. That replaces their `prior_pit`
charts with globally affine `affine_normal` charts (Gaussian δ prior, constant
`dδ/dz`). Off-mode curvature typically collapses by many orders of magnitude:


In [ ]:
# An explicit identically_linear= list *replaces* the auto-derived set.
# Union with ctx.identically_linear so DM/jumps/offsets stay certified.
declared = sorted(set(ctx_cert.identically_linear) | {"F0", "F1", "RAJ", "DECJ"})
ntm_lin = NonLinearTimingModel(
    engines="jug",
    inference="all",
    identically_linear=declared,
    name="timing",
)
ctx_lin = ntm_lin.for_pulsar(mp)

# Only F0/F1/RAJ/DECJ flip; the DM family stays affine_normal:
charts_default = {d["name"]: d["chart"] for d in ctx_cert.chart_summary()}
charts_lin = {d["name"]: d["chart"] for d in ctx_lin.chart_summary()}
print("axis    default chart   ->  declared chart")
for name in ("F0", "F1", "RAJ", "DECJ", "DM", "DM1"):
    flag = "  <-- flipped" if charts_default[name] != charts_lin[name] else ""
    print(f"  {name:5s} {charts_default[name]:14s} ->  {charts_lin[name]:14s}{flag}")

# Rebuild the joint model and certify at the SAME hyper probes.
psl_lin = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
    *ctx_lin.discovery_signals(joint=True),
])
jm_lin = N.joint_model(psl_lin, ctx_lin, reference_noise=reference, fixed=nd)
report_lin = certify_joint_geometry(jm_lin, ctx_lin, hyper_points=hyper_points)


def _row(label, r):
    return (f"  {label:26s}  rms={r.max_residual_remainder_rms:11.4g}"
            f"  max_std/TOA={r.max_residual_remainder_standardized_toa:11.4g}"
            f"  xi_grad={r.max_xi_gradient_inf_norm:11.4g}"
            f"  H_eig=[{r.xi_hessian_eigen_min:10.4g}, {r.xi_hessian_eigen_max:11.4g}]"
            f"  id_spread={r.max_conditional_identity_spread:11.4g}")


print("\nSAME data, SAME probes — only the charts on F0/F1/RAJ/DECJ changed:")
print(_row("wide uniform PIT charts", report))
print(_row("declared identically_linear", report_lin))
shrink = report.xi_hessian_eigen_max / report_lin.xi_hessian_eigen_max
print(f"\n-> Hessian eigenvalue max shrank by ~{shrink:,.0f}x; the negative "
      f"eigenvalue (a saddle off the mode) is gone, and the Hessian is ~identity.")
print(f"   passed: PIT charts={report.passed} -> declared={report_lin.passed}   "
      f"(residual failures: {len(report.failures)} -> {len(report_lin.failures)})")


**Takeaway.** Certify cheaply *before* a long run. A `passed=False` with runaway
metrics means “change the model,” not “turn NUTS knobs.” Here the fix is
`identically_linear=`.

On this isolated simulated pulsar every timing parameter is exactly linear, so
declaring them yields a clean pass (Hessian ≈ identity). On real pulsars with
binary orbits and free red noise, the report may still fail after this fix —
genuine nonlinear parameters and timing↔noise cross-curvature remain. That is
useful information: it tells you what still limits the geometry (see notebook
**03** on J1640).


### Transport-center report and saving the geometry product

For each axis, is the conditioned center still inside the chart? Read the
**chart type first**: an `affine_normal` axis is interior for any finite center
(Gaussian mean shift); only a `prior_pit` axis has a finite boundary, and only
there is a large `|center_z|` a saturation warning.

`write_geometry_report` / `read_geometry_report` persist a standalone JSON+NPZ
pair (digest-verified), separate from any sampling run manifest.


In [5]:
axes = transport_center_report(ctx_cert, jm.transport, center)
for a in axes[:5]:
    print(f"{a.name:12s} chart={a.chart:14s} center_z={a.center_z:+.3f} "
          f"interior={a.interior} chart_ratio={a.local_chart_ratio}")

stem = workdir / "geometry_report"
json_path, npz_path = write_geometry_report(report, stem, overwrite=True)
reloaded = read_geometry_report(stem)
print("\nwrote", json_path.name, "+", npz_path.name,
      "| digest-verified roundtrip:", reloaded.model_fingerprint == report.model_fingerprint)


RAJ          chart=prior_pit      center_z=+0.000 interior=True chart_ratio=0.9999999958667124
DECJ         chart=prior_pit      center_z=-0.000 interior=True chart_ratio=0.9999999999688715
F0           chart=prior_pit      center_z=-0.045 interior=True chart_ratio=0.9989675415248926
F1           chart=prior_pit      center_z=+0.353 interior=True chart_ratio=0.939457008589444
DM           chart=affine_normal  center_z=-0.016 interior=True chart_ratio=None

wrote geometry_report.json + geometry_report.npz | digest-verified roundtrip: True


## 3. Aside: pivot-amplitude red noise

When you sample a power-law red-noise amplitude at `f = 1/yr`, amplitude and
spectral slope are strongly correlated. Sampling the amplitude at a
**sensitivity-weighted pivot frequency** decorrelates them via an affine,
unit-Jacobian map:

`log10_A(1/yr) = log10_A_pivot + ½ γ log₁₀(f_pivot / f_ref)`.

The pivot frequency is computed from the Fourier basis and the white-noise
reference metric. This is independent of timing sampling, but it pairs well
with joint timing+noise runs.


In [6]:
NCOMP = 10
f, df, fmat = ds.fourierbasis(mp, NCOMP)
weights = ds.fourier_sensitivity_weights(fmat, dst.reference_noise(mp))
f_pivot = ds.sensitivity_weighted_pivot_frequency(np.asarray(f)[0::2], weights)
print(f"sensitivity-weighted pivot: {f_pivot:.3e} Hz  (1/yr = {ds.const.fyr:.3e} Hz)")

# Same spectrum, reparameterized: log10_A_pivot decoded to the reference amplitude
# reproduces the classic power law exactly.
param = ds.PowerLawParameterization(slope_pivot_frequency=f_pivot)
pl_pivot = ds.make_powerlaw_pivot(f_pivot=f_pivot, parameterization=param)
pl_ref = ds.make_powerlaw()
log10_A_pivot, gamma = -14.3, 3.7
log10_A_ref = ds.reference_log10_amplitude(
    log10_A_pivot, gamma, f_pivot=f_pivot, parameterization=param)
ff = np.array([1e-9, 3e-9, 1e-8])
dff = np.full_like(ff, 1e-9)
same = np.allclose(pl_pivot(ff, dff, log10_A_pivot, gamma),
                   pl_ref(ff, dff, log10_A_ref, gamma), rtol=1e-12)
print("pivot spectrum == reference spectrum:", same,
      f"| log10_A(1/yr) = {log10_A_ref:.4f}")


sensitivity-weighted pivot: 2.016e-08 Hz  (1/yr = 3.169e-08 Hz)
pivot spectrum == reference spectrum: True | log10_A(1/yr) = -14.6634


### Drop the pivot PSD into `makegp_fourier`

`make_powerlaw_pivot` is a drop-in PSD: the sampled amplitude is named
`..._log10_A_pivot` (at `f_pivot`) instead of `..._log10_A` (at 1/yr). Decode
back to 1/yr with `reference_log10_amplitude` for plots and papers.

When you pass a pivot GP to `N.joint_model` / `N.nuts`, supply explicit `priors=`
for the pivot amplitude and gamma — the default 1/yr amplitude prior does not
attach automatically to the pivot suffix.


In [7]:
rn_pivot = ds.makegp_fourier(mp, pl_pivot, NCOMP, name="rednoise_pivot")
psl_pivot = ds.PulsarLikelihood([
    mp.residuals,
    ds.makenoise_measurement_simple(mp, nd),
    rn_pivot,
])
pivot_params = [p for p in psl_pivot.logL.params if "rednoise_pivot" in p]
print("pivot RN sampled params:", pivot_params)
print("-> amplitude is sampled at f_pivot as '...rednoise_pivot_log10_A_pivot';")
print("   decode to 1/yr with reference_log10_amplitude(..., f_pivot=f_pivot).")


pivot RN sampled params: ['J1748-2021_rednoise_pivot_gamma', 'J1748-2021_rednoise_pivot_log10_A_pivot']
-> amplitude is sampled at f_pivot as '...rednoise_pivot_log10_A_pivot';
   decode to 1/yr with reference_log10_amplitude(..., f_pivot=f_pivot).
